You’re given a table of Uber rides that contains the mileage and the purpose for the business expense. You’re asked to find business purposes that generate the most miles driven for passengers that use Uber for their business transportation. Find the top 3 business purpose categories by total mileage.

In [0]:
CREATE TABLE ska_catalog.bronze.my_uber_drives (start_date TIMESTAMP,end_date TIMESTAMP,category VARCHAR(50),start VARCHAR(50),stop VARCHAR(50),miles FLOAT,purpose VARCHAR(50));

INSERT INTO ska_catalog.bronze.my_uber_drives (start_date, end_date, category, start, stop, miles, purpose) VALUES('2016-01-01 21:11', '2016-01-01 21:17', 'Business', 'Fort Pierce', 'Fort Pierce', 5.1, 'Meal/Entertain'),('2016-01-02 01:25', '2016-01-02 01:37', 'Business', 'Fort Pierce', 'Fort Pierce', 5, NULL),('2016-01-02 20:25', '2016-01-02 20:38', 'Business', 'Fort Pierce', 'Fort Pierce', 4.8, 'Errand/Supplies'),('2016-01-05 17:31', '2016-01-05 17:45', 'Business', 'Fort Pierce', 'Fort Pierce', 4.7, 'Meeting'),('2016-01-06 14:42', '2016-01-06 15:49', 'Business', 'Fort Pierce', 'West Palm Beach', 63.7, 'Customer Visit'),('2016-01-06 17:15', '2016-01-06 17:19', 'Business', 'West Palm Beach', 'West Palm Beach', 4.3, 'Meal/Entertain'),('2016-01-06 17:30', '2016-01-06 17:35', 'Business', 'West Palm Beach', 'Palm Beach', 7.1, 'Meeting');

In [0]:
SELECT * FROM ska_catalog.bronze.my_uber_drives

In [0]:
-- You’re given a table of Uber rides that contains the mileage and the purpose for the business expense. You’re asked to find business purposes that generate the most miles driven for passengers that use Uber for their business transportation. Find the top 3 business purpose categories by total mileage.
SELECT purpose AS `PURPOSE`,ROUND(SUM(miles),4) AS `TOTAL_MILES` 
FROM ska_catalog.bronze.my_uber_drives
WHERE category = 'Business'
GROUP BY PURPOSE
ORDER BY TOTAL_MILES DESC
LIMIT 3

# Intermediate to Advanced SQL Practice Questions

1. Find the top 3 longest rides by miles for each start location using window functions.

2. Calculate the cumulative miles traveled over time, ordered by `start_date`.

3. Determine the percentage contribution of each `purpose` to the total miles traveled.

4. Identify rides that lasted more than 1 hour and calculate their duration in minutes.

5. Rank all `purpose` values by their average miles per ride.

6. Find all rides where `miles` are greater than the overall average miles of all rides.

7. Show the difference in miles between consecutive rides (ordered by `start_date`).

8. Create a pivot-style result showing total miles grouped by `category` and `purpose`.

9. Find the most frequent `start-stop` pair and the number of times it occurred.

10. Detect duplicate rides based on the same `start_date` and `start-stop` pair.

---

11. Using a CTE, calculate the average miles per purpose and then filter purposes with an average greater than 10 miles.

12. Find the ride with the maximum duration and display its start, stop, and duration in minutes.

13. Show the top 5 purposes contributing the most to total miles using a subquery.

14. Calculate the running average of miles over time using window functions.

15. Find the difference between the longest and shortest ride in terms of miles.

16. Display rides grouped by day and show total miles per day.

17. Identify the category that has the highest number of rides and its count.

18. For each start location, find the average duration of rides and rank them by this average.

19. Create a query that returns the start location with the highest cumulative miles using window functions.

20. Find all rides where the purpose is missing (`NULL`) and replace it with 'Unknown' in the result set.

In [0]:
-- Find the top 3 longest rides by miles for each start location using window functions.
SELECT * FROM (
SELECT *,rank() OVER (PARTITION BY start ORDER BY miles DESC) AS `rank`
FROM ska_catalog.bronze.my_uber_drives)
WHERE rank <= 3

In [0]:
-- Calculate the cumulative miles traveled over time, ordered by start_date.
SELECT start_date,category,start,miles, SUM(miles) OVER (ORDER BY start_date ) AS `Cumulative_miles`
FROM ska_catalog.bronze.my_uber_drives
ORDER BY start_date

In [0]:
-- Determine the percentage contribution of each purpose to the total miles traveled.
SELECT 
  purpose,
  ROUND(SUM(miles), 4) AS total_miles,
  ROUND(100 * SUM(miles) / SUM(SUM(miles)) OVER (), 2) AS percent_contribution
FROM ska_catalog.bronze.my_uber_drives
GROUP BY purpose
ORDER BY percent_contribution DESC

In [0]:
-- Identify rides that lasted more than 1 hour and calculate their duration in minutes.

SELECT 
  start_date,
  end_date,
  start,
  stop,
  purpose,
  miles,
  ROUND((unix_timestamp(end_date) - unix_timestamp(start_date)) / 60, 2) AS duration_minutes
FROM ska_catalog.bronze.my_uber_drives
WHERE (unix_timestamp(end_date) - unix_timestamp(start_date)) / 60 > 60

In [0]:
-- Rank all purpose values by their average miles per ride.
SELECT
  purpose,
  ROUND(AVG(miles), 4) AS avg_miles_per_ride,
  RANK() OVER (ORDER BY AVG(miles) DESC) AS purpose_rank
FROM ska_catalog.bronze.my_uber_drives
GROUP BY purpose
ORDER BY purpose_rank

-- Rank all purpose values by their average miles per ride.
-- 1. FROM ska_catalog.bronze.my_uber_drives
-- 2. GROUP BY purpose
-- 3. SELECT purpose, ROUND(AVG(miles), 4) AS avg_miles_per_ride
-- 4. Compute AVG(miles) for each purpose
-- 5. Compute RANK() OVER (ORDER BY AVG(miles) DESC) AS purpose_rank
-- 6. ORDER BY purpose_rank

In [0]:
-- Find all rides where the purpose is missing (NULL) and replace it with 'Unknown' in the result set.
SELECT
  start_date,
  end_date,
  category,
  start,
  stop,
  miles,
  COALESCE(purpose, 'Unknown') AS purpose
FROM ska_catalog.bronze.my_uber_drives

In [0]:
-- Return the start location with the highest cumulative miles using window functions.
SELECT start AS start_location, total_miles
FROM (
  SELECT 
    start,
    SUM(miles) AS total_miles,
    RANK() OVER (ORDER BY SUM(miles) DESC) AS rnk
  FROM ska_catalog.bronze.my_uber_drives
  GROUP BY start
) ranked
WHERE rnk = 1